# Numerical companion of Section 6 (approximate gradient information from finite sums)

This notebook contains the numerical computations behind Section 6 of *Accelerated gradient methods in unconstrained convex optimization* (d'Aspremont, Kunstner, Taylor); the symbolic verification of Theorem 6.1 is in `Section6_symbolics.ipynb`. The contraction factor of Theorem 6.1 (SAGA) is compared with the exact one, computed with [PEPit](https://pepit.readthedocs.io).

In [1]:
!pip install -q pepit==0.5.1

In [2]:
import numpy as np

from PEPit import PEP
from PEPit.functions import SmoothStronglyConvexFunction

## SAGA: the contraction factor

For $f=\tfrac1n\sum_{i=1}^nf_i$ with $f_i\in\mathcal F_{m,L}$, SAGA keeps the stored points $\phi_{i,k}$, uses $d_k=\nabla f_{i_k}(x_k)-\nabla f_{i_k}(\phi_{i_k,k})+\tfrac1n\sum_{j=1}^n\nabla f_j(\phi_{j,k})$, and sets $x_{k+1}=x_k-\alpha\,d_k$ and $\phi_{i_k,k+1}=x_k$. With $B_i(x)=f_i(x)-f_i(x_\star)-\langle\nabla f_i(x_\star),x-x_\star\rangle$ and the potential $\Phi_k=\tfrac1n\sum_{i=1}^nB_i(\phi_{i,k})+c\|x_k-x_\star\|^2$, $c=\tfrac1{2\alpha(1-\alpha m)n}$, Theorem 6.1 states $\mathbb E_k[\Phi_{k+1}]\leq\big(1-\min\{\tfrac1{4n},\tfrac m{3L}\}\big)\Phi_k$ for $\alpha=\tfrac1{3L}$.

The exact one-iteration contraction factor, $\max\{\mathbb E_k[\Phi_{k+1}]:\Phi_k\leq1\}$, is itself a performance estimation problem, over $n$ functions of the class and all possible states $(x_k,\phi_{1,k},\ldots,\phi_{n,k})$, with the expectation written as the average over the $n$ possible values of the sampled index. The function below computes it with PEPit, and returns it together with the factor of the theorem.

In [3]:
def wc_saga(n, L, m, alpha, verbose=0):
    # Instantiate PEP
    problem = PEP()

    # Declare n smooth strongly convex functions
    fn = [problem.declare_function(SmoothStronglyConvexFunction,
                                   L=L, mu=m, reuse_gradient=True) for _ in range(n)]

    # Define the objective as the average of the former
    func = np.mean(fn)

    # Start by defining its unique optimal point xs = x_*
    xs = func.stationary_point()

    # Then define the initial points
    phi = [problem.set_initial_point() for _ in range(n)]
    x0 = problem.set_initial_point()

    # Compute the initial value of the Lyapunov function, for a given parameter
    c = 1 / 2 / alpha / (1 - m * alpha) / n
    g, f = [x0 for _ in range(n)], [x0 for _ in range(n)]
    g0, f0 = [x0 for _ in range(n)], [x0 for _ in range(n)]
    gs, fs = [x0 for _ in range(n)], [x0 for _ in range(n)]
    init_lyapunov = c * (xs - x0) ** 2

    for i in range(n):
        g[i], f[i] = fn[i].oracle(phi[i])
        gs[i], fs[i] = fn[i].oracle(xs)
        init_lyapunov = init_lyapunov + 1 / n * (f[i] - fs[i] - gs[i] * (phi[i] - xs))

    # Set the initial constraint as the Lyapunov bounded by 1
    problem.set_initial_condition(init_lyapunov <= 1)

    # Compute the expected value of the Lyapunov function after one iteration
    # (so: expectation over n possible scenarios: one for each element fi in the function).
    final_lyapunov_avg = (xs - xs) ** 2
    for i in range(n):
        g0[i], f0[i] = fn[i].oracle(x0)
        x1 = x0 - alpha * (g0[i] - g[i])
        for j in range(n):
            x1 = x1 - alpha / n * g[j]
        final_lyapunov = c * (x1 - xs) ** 2
        for j in range(n):
            if i != j:
                gi, fi = g[j], f[j]
                final_lyapunov = final_lyapunov + 1 / n * (fi - fs[j] - gs[j] * (phi[j] - xs))
            else:
                gi, fi = g0[i], f0[i]
                final_lyapunov = final_lyapunov + 1 / n * (fi - fs[j] - gs[j] * (x0 - xs))
        final_lyapunov_avg = final_lyapunov_avg + final_lyapunov / n

    # Set the performance metric to the expected value of the Lyapunov function after one iteration
    problem.set_performance_metric(final_lyapunov_avg)

    # Solve the PEP
    pepit_verbose = max(verbose, 0)
    pepit_tau = problem.solve(verbose=pepit_verbose)

    # Compute theoretical guarantee of Theorem 6.1 (for comparison; it holds for alpha = 1/(3L))
    theoretical_tau = 1 - min(1 / (4 * n), m / (3 * L))

    # Print conclusion if required
    if verbose != -1:
        print('*** Example file: worst-case performance of SAGA for Lyapunov function V ***')
        print('\tPEPit guarantee:\t V^(k+1) <= {:.6} V^(k)'.format(pepit_tau))
        print('\tTheoretical guarantee:\t V^(k+1) <= {:.6} V^(k)'.format(theoretical_tau))

    # Return the worst-case guarantee of the evaluated method (and the reference theoretical value)
    return pepit_tau, theoretical_tau

In [4]:
L, m, n = 1, .1, 20
alpha = 1 / 3 / L
pepit_tau, theoretical_tau = wc_saga(n, L, m, alpha)

[pepit_tau, theoretical_tau]

*** Example file: worst-case performance of SAGA for Lyapunov function V ***
	PEPit guarantee:	 V^(k+1) <= 0.972532 V^(k)
	Theoretical guarantee:	 V^(k+1) <= 0.9875 V^(k)


[0.97253213101599, 0.9875]

The factor of the theorem is an upper bound on the exact one. The next cell compares the two for a few values of $n$ and $m$ (with $L=1$), together with the factor $\max\big\{1-\tfrac1n\big(1-\tfrac{1}{2(1-\alpha m)}\big),\,1-\alpha m\big\}$ that the proof of the theorem gives before its last simplification, and with $1-\tfrac m{3L}$.

In [5]:
L = 1
alpha = 1 / 3 / L

print('  n     m     PEPit   Theorem 6.1   proof, before simplification   1 - m/(3L)')
for n, m in [(2, 0.05), (3, 0.05), (2, 0.2), (10, .1), (20, .1)]:
    pepit_tau, theoretical_tau = wc_saga(n, L, m, alpha, verbose=-1)
    proof_tau = max(1 - (1 - 1 / (2 * (1 - alpha * m))) / n, 1 - alpha * m)
    print(f'{n:3d}  {m:4.2f}   {pepit_tau:.5f}     {theoretical_tau:.5f}               {proof_tau:.5f}               {1 - m / (3 * L):.5f}')

  n     m     PEPit   Theorem 6.1   proof, before simplification   1 - m/(3L)
  2  0.05   0.98333     0.98333               0.98333               0.98333
  3  0.05   0.98333     0.98333               0.98333               0.98333
  2  0.20   0.93333     0.93333               0.93333               0.93333
 10  0.10   0.96667     0.97500               0.96667               0.96667
 20  0.10   0.97253     0.98750               0.97586               0.96667


Two regimes appear, according to the term that is active in the minimum:

- when $\tfrac m{3L}\leq\tfrac1{4n}$ (first three rows), the factor of the theorem is the exact one. Indeed, $1-\tfrac m{3L}$ is a lower bound on the exact factor for every $n$: it is the value of $\mathbb E_k[\Phi_{k+1}]/\Phi_k$ for $f_i(x)=\tfrac m2\|x-x_\star\|^2$ with all the stored points at $x_\star$;
- when $\tfrac m{3L}>\tfrac1{4n}$ (last two rows), the theorem is conservative. For $n=10$, the exact factor is still $1-\tfrac m{3L}$, and for $n=20$ it is about $1-\tfrac{0.55}{n}$, to be compared with $1-\tfrac1{4n}$. Part of the difference comes from the simplification $1-\tfrac1{2(1-\alpha m)}\geq\tfrac14$ at the end of the proof, as shown by the column giving the factor of the proof before this simplification.